# A disproportionality signal from public adverse event reports

**Data kit lab** `faers-ticagrelor-dyspnoea-ror` · Safety · about 30 minutes

**The question.** Is dyspnoea reported more often with ticagrelor than with all other drugs in the US FDA Adverse Event Reporting System (FAERS)?

**What you do.**

1. Ask the openFDA API how many reports were first received in a window, and read its notes on the data.
2. Count the reports that list ticagrelor as a suspect drug.
3. Count the reports with the reaction term Dyspnoea, for ticagrelor and for all drugs.
4. Build the 2x2 table and compute the reporting odds ratio (ROR), its 95% confidence interval, the proportional reporting ratio (PRR) and chi-squared.
5. Compare your numbers with a published analysis of the same database: Pan and others, Frontiers in Pharmacology 2024 (DOI 10.3389/fphar.2024.1251961), Table 2, row Dyspnea.

**Read this first.** These are spontaneous reports. openFDA says they are not verified for cause and effect and cannot be used to estimate incidence. A disproportionality measure compares shares of reports. It is not an incidence, and it does not show that a drug causes an event. It flags a pair for review.

**The paper used another route.** It ran OpenVigil 2.1, counted only reports where ticagrelor was the *primary* suspect, mapped drug names by its own rules, and counted whole cases across report versions. The public API cannot apply all of that. This lab applies the closest method the API supports and shows where the numbers part. Expect them to differ.

Data: openFDA (FDA), CC0 1.0 unless otherwise noted. The MedDRA dictionary itself is not used; reaction terms are read as they appear in the reports.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'faers-ticagrelor-dyspnoea-ror'
FILES = json.loads(r'''{
 "all-window": {
  "url": "https://api.fda.gov/drug/event.json?search=receivedate:[20101001+TO+20230331]&limit=1",
  "name": "all-window.json",
  "sha256": "cb19fcb9e78b7534d3064131f5e1a9f2fe83359786f0c06616e5280729341328",
  "bytes": 2636
 },
 "drug-window": {
  "url": "https://api.fda.gov/drug/event.json?search=receivedate:[20101001+TO+20230331]+AND+patient.drug.openfda.generic_name:ticagrelor+AND+patient.drug.drugcharacterization:1&limit=1",
  "name": "drug-window.json",
  "sha256": "73c64c4ecd460a0317e45b6e80eeb45279e574cf301a752d83995dcd7dc83d23",
  "bytes": 88674
 },
 "drug-reactions": {
  "url": "https://api.fda.gov/drug/event.json?search=receivedate:[20101001+TO+20230331]+AND+patient.drug.openfda.generic_name:ticagrelor+AND+patient.drug.drugcharacterization:1&count=patient.reaction.reactionmeddrapt.exact",
  "name": "drug-reactions.json",
  "sha256": "aa05e6e374f5c9df7dcb5b9bae16df7087ba3396ed8056727adf416fc9422492",
  "bytes": 6970
 },
 "all-reactions": {
  "url": "https://api.fda.gov/drug/event.json?search=receivedate:[20101001+TO+20230331]&count=patient.reaction.reactionmeddrapt.exact",
  "name": "all-reactions.json",
  "sha256": "933ee3207b2bb21698d20f8f6468dd834c3fc9107644bc52160c377c747a941d",
  "bytes": 7157
 }
}''')
PUBLISHED = json.loads(r'''{
 "n_reports_window": {
  "label": "All reports in the window (N)",
  "value": 9699440,
  "tolerance": {
   "exact": true
  }
 },
 "n_drug_reports": {
  "label": "Reports with ticagrelor (a+b)",
  "value": 12909,
  "tolerance": {
   "exact": true
  }
 },
 "a_drug_event": {
  "label": "Reports with ticagrelor and dyspnoea (a)",
  "value": 1824,
  "tolerance": {
   "exact": true
  }
 },
 "ror": {
  "label": "Reporting odds ratio",
  "value": 7.34,
  "tolerance": {
   "abs": 0.005
  }
 },
 "ror_ci_low": {
  "label": "ROR 95% CI lower",
  "value": 6.99,
  "tolerance": {
   "abs": 0.005
  }
 },
 "ror_ci_high": {
  "label": "ROR 95% CI upper",
  "value": 7.72,
  "tolerance": {
   "abs": 0.005
  }
 },
 "prr": {
  "label": "Proportional reporting ratio",
  "value": 6.45,
  "tolerance": {
   "abs": 0.005
  }
 },
 "chi2": {
  "label": "Chi-squared",
  "value": 8505.6,
  "tolerance": {
   "abs": 0.005
  }
 }
}''')
PUBLISHED_FROM = 'Europe PMC full-text record PMC11035729 (XML, no page numbers): Results section 3.1; Table 2, PT signal detection results of the top 50 ADEs, row Dyspnea | https://www.ebi.ac.uk/europepmc/webservices/rest/PMC11035729/fullTextXML'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. All reports in the window

Each file in this lab is one API call. The window is 1 October 2010 to 31 March 2023, the paper's window. The paper does not say which date it used; this lab uses `receivedate`, the date FDA first received a report. It was fixed before any number was read.

Every openFDA answer carries a `meta` block with the date of the data and a disclaimer. Read it before the numbers.

In [ ]:
import json
import math
from urllib.parse import unquote

import pandas as pd


def load(file_id):
    return json.loads(data_file(file_id).read_text())


def query(file_id):
    """The search and count part of a file's URL, made readable."""
    return unquote(FILES[file_id]['url'].split('?', 1)[1]).replace('+', ' ')


all_window = load('all-window')
meta = all_window['meta']
print('query:', query('all-window'))
print('data last updated:', meta['last_updated'])
print('terms:', meta['terms'])
print('license:', meta['license'])
print('disclaimer:', meta['disclaimer'][:300], '...')
N = meta['results']['total']
print(f'N, all reports first received in the window: {N:,}')

## 2. Reports that list ticagrelor as a suspect

The search asks for reports with the harmonised generic name `ticagrelor` and a drug with role 1. The field reference lists three roles: 1 suspect, 2 concomitant, 3 interacting. There is no separate code for the primary suspect, so this count cannot match the paper's filter.

Each drug entry in a report has its own name and role. Whether this search can match the name on one entry and the role on another is not tested in this lab (see Try next).

In [ ]:
drug_window = load('drug-window')
print('query:', query('drug-window'))
n_drug = drug_window['meta']['results']['total']
print(f'reports with ticagrelor and a suspect drug (a + b): {n_drug:,}')

## 3. Reports with the term Dyspnoea

The `count` parameter returns the most frequent values of a field, with the number of matching reports for each. Terms are stored in British English, so the term is Dyspnoea; the paper prints Dyspnea. Match the whole term. A prefix match would also catch terms such as Dyspnoea exertional.

The query-parameter page says `count` returns the 1000 most frequent values. Count how many terms came back.

In [ ]:
def term_table(file_id):
    d = load(file_id)
    t = pd.DataFrame(d['results']).rename(columns={'term': 'reaction_term', 'count': 'reports'})
    t.index = t.index + 1
    return t


drug_terms = term_table('drug-reactions')
all_terms = term_table('all-reactions')
print('terms returned: ticagrelor reports', len(drug_terms), '| all reports', len(all_terms))
print(drug_terms.head(10).to_string())

TERM = 'dyspnoea'
print()
print('terms that start with', TERM, 'in the ticagrelor list:')
print(drug_terms[drug_terms['reaction_term'].str.lower().str.startswith(TERM)].to_string())


def count_of(table, term):
    hit = table[table['reaction_term'].str.lower() == term]
    if len(hit) != 1:
        raise SystemExit(f'{term!r} is not in the list of {len(table)} terms returned; the count cannot be read')
    return int(hit['reports'].iloc[0])


a = count_of(drug_terms, TERM)
n_event = count_of(all_terms, TERM)
print()
print(f'a, ticagrelor reports with Dyspnoea: {a:,}')
print(f'a + c, all reports with Dyspnoea: {n_event:,}')

## 4. The 2x2 table and the statistics

The paper's Table 1 gives the formulas. Its equations put the drug's reports in the first row:

| | Dyspnoea | Other terms |
|---|---|---|
| Ticagrelor | a | b |
| All other drugs | c | d |

ROR = (a/c)/(b/d), with SE(ln ROR) = sqrt(1/a + 1/b + 1/c + 1/d) and 95% CI = exp(ln ROR ± 1.96 SE). PRR = [a/(a+b)] / [c/(c+d)]. Chi-squared = (ad − bc)² (a+b+c+d) / [(a+b)(c+d)(a+c)(b+d)], with no continuity correction.

The footnote of Table 1 describes b and c the other way round. ROR and chi-squared do not change if you swap them; PRR does. Section 5 checks which reading the paper's printed numbers follow.

In [ ]:
def stats(a, b, c, d):
    ror = (a / c) / (b / d)
    se = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    n = a + b + c + d
    return {
        'ror': ror,
        'ror_ci_low': math.exp(math.log(ror) - 1.96 * se),
        'ror_ci_high': math.exp(math.log(ror) + 1.96 * se),
        'prr': (a / (a + b)) / (c / (c + d)),
        'chi2': (a * d - b * c) ** 2 * n / ((a + b) * (c + d) * (a + c) * (b + d)),
    }


b = n_drug - a
c = n_event - a
d = N - n_drug - c
table = pd.DataFrame({'Dyspnoea': [a, c], 'Other terms': [b, d]}, index=['Ticagrelor', 'All other drugs'])
print(table.to_string())
print(f'total {table.values.sum():,} (N = {N:,})')
S = stats(a, b, c, d)
for k, v in S.items():
    print(f'{k:<12}{v:,.2f}')

## 5. The paper's own arithmetic

The paper prints a, a + b and N, but not c. You can solve for c from its printed ROR and then recompute PRR and chi-squared both ways. This uses only the paper's numbers; nothing here is checked.

In [ ]:
P = {k: v['value'] for k, v in PUBLISHED.items()}
pa, pn_drug, pN = P['a_drug_event'], P['n_drug_reports'], P['n_reports_window']
pb = pn_drug - pa


def ror_for(c):
    return (pa / c) / (pb / (pN - pn_drug - c))


lo, hi = 1.0, float(pN - pn_drug - 1)  # ror_for falls as c grows
for _ in range(200):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if ror_for(mid) > P['ror'] else (lo, mid)
pc = (lo + hi) / 2
pd_ = pN - pn_drug - pc
eq = stats(pa, pb, pc, pd_)
foot_prr = (pa / (pa + pc)) / (pb / (pb + pd_))
print(f'implied c (Dyspnoea reports with other drugs): about {pc:,.0f}')
print(f"equations' reading: PRR {eq['prr']:.2f}, chi2 {eq['chi2']:,.1f}, 95% CI {eq['ror_ci_low']:.2f} to {eq['ror_ci_high']:.2f}")
print(f"footnote's reading: PRR {foot_prr:.2f}")
print(f"printed:            PRR {P['prr']}, chi2 {P['chi2']:,}, 95% CI {P['ror_ci_low']} to {P['ror_ci_high']}")

## 6. Compare with the paper

The checks below compare each count and statistic with the paper. Counts must match exactly; ratios within half a unit of the last printed digit. The method differences from the top of this notebook are not folded into the tolerance, so a difference here is the result, not an error.

In [ ]:
RESULTS = {
    'n_reports_window': N,
    'n_drug_reports': n_drug,
    'a_drug_event': a,
    **S,
}
for k in ['n_reports_window', 'n_drug_reports', 'a_drug_event']:
    print(f'{k:<18} this lab / paper = {RESULTS[k] / P[k]:.2f}')
print(f"share of the drug's reports with Dyspnoea: this lab {a / n_drug:.3f}, paper {pa / pn_drug:.3f}")
report(RESULTS)

## What the result says

Look at the three counts first, and at the ratios printed above the table. Which count differs most from the paper? The paper's filter was narrower in three ways: primary suspect only, its own drug-name mapping, and an extract that ended in March 2023. Is the share of the drug's reports that carry Dyspnoea closer between the two than the raw counts are?

Then read the ratios. Do both analyses put the lower limit of the ROR's 95% CI above 1? The paper's Table 1 states its signal rules (for ROR, a > 3 and a lower limit above 1; for PRR, a > 3, PRR > 2 and chi-squared > 4). Apply them to your numbers.

A pair that passes these rules is a signal to review, not a finding. It does not give the rate of dyspnoea in people who take ticagrelor, and it does not show that ticagrelor causes it.

## Try next

1. Change `receivedate` to `receiptdate` in the URLs (a new file id each) and note how much the counts move. Write down which one you would use, and why, before you look.
2. The count calls here returned fewer terms than the 1000 the documentation describes. Add `&limit=1000` to a count URL and see how many come back.
3. Is a report counted where ticagrelor is only concomitant and another drug is the suspect? Fetch a few ticagrelor reports with `limit=100` and look at the `drugcharacterization` of the ticagrelor entry itself.
4. Read the paper's Methods 2.1 and list each filter it applied. For each one, write down whether the openFDA API can apply it.
5. Repeat the table for another row of the paper's Table 2, such as Contusion or Hemorrhage. Find the British spelling in the count list first.